In [60]:
from typing import TypedDict, List
from langgraph.graph import StateGraph, START, END
from IPython.display import Image, display
import random

class AgentState(TypedDict):
    magic_number: int
    guesses: List[int]
    attempts: int
    limit: int
    lower_bound: int
    upper_bound: int
    hint: str
    result: str

def setup_node(state: AgentState) -> AgentState:
    magic_number = random.randint(state['lower_bound'], state['upper_bound'])
    print(f"[SETUP] Magic number defined: {magic_number}")

    state['magic_number'] = magic_number
    state['attempts'] = 0
    state['limit'] = 7
    state['result'] = 'lost'
    return state

def guess_node(state: AgentState) -> AgentState:
    print(f"[GUESS] Running guess: hint={state['hint']}")

    if state['hint'] == 'lower':
        last_guess = state["guesses"][-1]
        state['upper_bound'] = last_guess - 1
        print(f"[GUESS] Had to lower the upper_bound: {state['upper_bound']}")

    if state['hint'] == 'higher':
        last_guess = state["guesses"][-1]
        state['lower_bound'] = last_guess + 1
        print(f"[GUESS] Had to up the lower_bound: {state['lower_bound']}")
        
    guess = random.randint(state['lower_bound'], state['upper_bound'])
    state['guesses'].append(guess)
    state['attempts'] = state['attempts'] + 1

    return state

def hint_node(state: AgentState) -> AgentState:
    print(f"[HINT] Running hint: guesses={state['guesses']}")
    last_guess = state["guesses"][-1]
    
    if last_guess == state['magic_number']:
        print(f"[HINT] Player won the game")
        state['result'] = 'won'
    if last_guess > state['magic_number']:
        state['hint'] = 'lower'
    elif last_guess < state['magic_number']:
        state['hint'] = 'higher'
    
    return state

def validate_win_node(state: AgentState) -> AgentState:
    last_guess = state["guesses"][-1]
    print(f"[VALIDATE_WIN] Validating: result={state['result']}, magic_number={state['magic_number']}, last_guess={last_guess}, attempts={state['attempts']}, limit={state['limit']}")

    if state['attempts'] == state['limit'] or state['result'] == 'won':
        return 'finish'

    if state['attempts'] < state['limit']:
        return 'continue'
    

    

In [66]:
graph = StateGraph(AgentState)
graph.add_node('setup', setup_node)
graph.add_node('guess', guess_node)
graph.add_node('hint', hint_node)

graph.set_entry_point('setup')
graph.add_edge('setup', 'guess')
graph.add_edge('guess', 'hint')
graph.add_conditional_edges(
    "hint",
    validate_win_node,
    {
        'continue': 'guess',
        'finish': END,
    }
)

graph.add_node('validate_win', lambda state: state)

app = graph.compile()
# display(Image(app.get_graph().draw_mermaid_png()))

state = AgentState(
    guesses=[],
    lower_bound=1,
    upper_bound=100,
    hint=''
)

result = app.invoke(state)
result


[SETUP] Magic number defined: 55
[GUESS] Running guess: hint=
[HINT] Running hint: guesses=[78]
[VALIDATE_WIN] Validating: result=lost, magic_number=55, last_guess=78, attempts=1, limit=7
[GUESS] Running guess: hint=lower
[GUESS] Had to lower the upper_bound: 77
[HINT] Running hint: guesses=[78, 22]
[VALIDATE_WIN] Validating: result=lost, magic_number=55, last_guess=22, attempts=2, limit=7
[GUESS] Running guess: hint=higher
[GUESS] Had to up the lower_bound: 23
[HINT] Running hint: guesses=[78, 22, 60]
[VALIDATE_WIN] Validating: result=lost, magic_number=55, last_guess=60, attempts=3, limit=7
[GUESS] Running guess: hint=lower
[GUESS] Had to lower the upper_bound: 59
[HINT] Running hint: guesses=[78, 22, 60, 53]
[VALIDATE_WIN] Validating: result=lost, magic_number=55, last_guess=53, attempts=4, limit=7
[GUESS] Running guess: hint=higher
[GUESS] Had to up the lower_bound: 54
[HINT] Running hint: guesses=[78, 22, 60, 53, 54]
[VALIDATE_WIN] Validating: result=lost, magic_number=55, last_gu

{'magic_number': 55,
 'guesses': [78, 22, 60, 53, 54, 58, 57],
 'attempts': 7,
 'limit': 7,
 'lower_bound': 55,
 'upper_bound': 57,
 'hint': 'lower',
 'result': 'lost'}